# 05 - Quality Report And Analytics

This notebook produces portfolio-friendly outputs: record counts, data quality score, and business analytics from the Gold layer.

In [ ]:
%%configure -f
{
  "conf": {
    "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension",
    "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"
  }
}

In [ ]:
from datetime import datetime

BRONZE_PATH = "s3://loanshield-bronze/"
SILVER_PATH = "s3://loanshield-silver/"
GOLD_PATH = "s3://loanshield-gold/"
REJECTED_PATH = "s3://loanshield-rejected/"
REPORTS_PATH = "s3://loanshield-reports/"

df_bronze = spark.read.format("delta").load(BRONZE_PATH)
df_silver = spark.read.format("delta").load(SILVER_PATH)
df_gold = spark.read.format("delta").load(GOLD_PATH)
df_rejected = spark.read.format("delta").load(REJECTED_PATH)

bronze_count = df_bronze.count()
silver_count = df_silver.count()
gold_count = df_gold.count()
rejected_count = df_rejected.count()
quality_score = round((gold_count / bronze_count) * 100, 2) if bronze_count else 0

report_data = [
    ("run_date", str(datetime.now())),
    ("bronze_records", str(bronze_count)),
    ("silver_records", str(silver_count)),
    ("gold_records", str(gold_count)),
    ("rejected_records", str(rejected_count)),
    ("total_removed", str(bronze_count - gold_count)),
    ("data_quality_score", f"{quality_score}%"),
    ("pipeline_status", "SUCCESS")
]

df_report = spark.createDataFrame(report_data, ["metric", "value"])
report_path = f"{REPORTS_PATH.rstrip('/')}/quality_report_{datetime.now().strftime('%Y_%m_%d_%H_%M_%S')}"
df_report.write.format("csv").mode("overwrite").option("header", "true").save(report_path)

print(f"Quality report saved to: {report_path}")
df_report.show(truncate=False)

In [ ]:
df_gold.groupBy("risk_category").count().orderBy("risk_category").show(truncate=False)
df_gold.groupBy("fico_category").count().orderBy("fico_category").show(truncate=False)
df_gold.groupBy("loan_size_category").count().orderBy("loan_size_category").show(truncate=False)
df_gold.groupBy("dti_category").count().orderBy("dti_category").show(truncate=False)

In [ ]:
df_gold.createOrReplaceTempView("gold_loans")
spark.sql("""
    SELECT risk_category, COUNT(*) AS loan_count, ROUND(AVG(loan_amnt), 2) AS avg_loan_amount
    FROM gold_loans
    GROUP BY risk_category
    ORDER BY risk_category
""").show(truncate=False)